# Unified Comparison: MHA vs MQA vs GQA vs MLA

Compare all 4 attention mechanisms across TinyStories and SimpleStories datasets.

**Metrics:**
- Validation Perplexity & Loss
- Top-1/Top-5 Accuracy
- Parameter Count (total & non-embedding)
- KV-Cache Size (analytical)
- Generation Quality (side-by-side samples)
- Inference Speed (tokens/sec)
- Peak GPU Memory

**Models:** 4 mechanisms x 2 datasets = 8 models

## 1. Setup

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

# Install dependencies
!pip install -q transformers datasets tqdm tensorboard matplotlib pandas

In [ ]:
# Clone repository, setup path, patch __init__.py, import all modules
import sys, os, importlib, shutil, torch, gc

# Clone repository
if os.path.exists('PROJECT'):
    !rm -rf PROJECT
    print("Existing repo removed")
!git clone https://gitlab.cim.rhul.ac.uk/wmis066/PROJECT.git
print("Repository cloned")
%cd PROJECT

# Clear Python cache
print("Clearing Python cache...")
modules_to_remove = [m for m in list(sys.modules.keys())
                     if any(x in m for x in ['mla', 'mqa', 'gqa', 'mha', 'train', 'transformer', 'data_loader', 'attention', 'layers'])]
for module in modules_to_remove:
    del sys.modules[module]
print(f"Removed {len(modules_to_remove)} cached modules")

cache_dirs = [
    '/content/PROJECT/AttentionHeads/mha/__pycache__',
    '/content/PROJECT/AttentionHeads/mqa/__pycache__',
    '/content/PROJECT/AttentionHeads/gqa/__pycache__',
    '/content/PROJECT/AttentionHeads/mla/__pycache__',
    '/content/PROJECT/AttentionHeads/__pycache__',
]
for cache_dir in cache_dirs:
    if os.path.exists(cache_dir):
        shutil.rmtree(cache_dir)

project_root = '/content/PROJECT'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Patch __init__.py to comment out positional_encoding import
init_file_path = os.path.join(project_root, 'AttentionHeads', 'mha', '__init__.py')
if os.path.exists(init_file_path):
    with open(init_file_path, 'r') as f:
        lines = f.readlines()
    patched_lines = []
    modified = False
    in_block = False
    block_indent = -1
    for line in lines:
        stripped = line.strip()
        indent = len(line) - len(line.lstrip())
        if "from .positional_encoding import" in line:
            in_block = True
            block_indent = indent
            if not stripped.startswith('#'):
                patched_lines.append(f"# PATCHED: {stripped}\n")
                modified = True
            else:
                patched_lines.append(line)
            continue
        if in_block:
            if indent > block_indent or stripped == '' or (')' in stripped and indent <= block_indent):
                if not stripped.startswith('#'):
                    patched_lines.append(f"# PATCHED: {stripped}\n")
                    modified = True
                else:
                    patched_lines.append(line)
                if ')' in stripped and indent <= block_indent:
                    in_block = False
                continue
            else:
                in_block = False
        patched_lines.append(line)
    if modified:
        with open(init_file_path, 'w') as f:
            f.writelines(patched_lines)
        print("Patched __init__.py")

# Import all 4 transformer modules
from AttentionHeads.mha import transformer as mha_transformer
from AttentionHeads.mqa import transformer as mqa_transformer
from AttentionHeads.gqa import transformer as gqa_transformer
from AttentionHeads.mla import transformer as mla_transformer
from AttentionHeads.mha import data_loader, train

for mod in [mha_transformer, mqa_transformer, gqa_transformer, mla_transformer, data_loader, train]:
    importlib.reload(mod)

print("All modules imported successfully!")

## 2. Load All Models

In [ ]:
import torch
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import GPT2Tokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Base model config
base_config = {
    'vocab_size': 50257, 'hidden_size': 256, 'num_layers': 4, 'num_heads': 8,
    'intermediate_size': 1024, 'max_position_embeddings': 256, 'dropout': 0.2,
    'position_embedding_type': 'rope', 'activation': 'gelu',
    'layer_norm_epsilon': 1e-5, 'initializer_range': 0.02, 'tie_word_embeddings': True
}

# Model-specific configs
configs = {
    'MHA': {**base_config},
    'MQA': {**base_config},
    'GQA-4': {**base_config, 'num_kv_heads': 4},
    'MLA': {**base_config, 'd_c': 128, 'd_rope': 16},
}

# Create functions for each model type
create_fns = {
    'MHA': mha_transformer.create_gptneo_model,
    'MQA': mqa_transformer.create_gptneo_model,
    'GQA-4': gqa_transformer.create_gptneo_model,
    'MLA': mla_transformer.create_gptneo_model,
}

def load_model(name, checkpoint_path):
    """Load a model from checkpoint"""
    model = create_fns[name](configs[name])
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'], strict=False)
    model.to(device)
    model.eval()
    return model, checkpoint

print("Model configs and loading functions defined!")

In [ ]:
# Checkpoint paths (update these to your saved locations)
# TinyStories models
ts_paths = {
    'MHA': '/content/drive/MyDrive/GPTNeo_TinyStories/gptneo_mha_tinystories/best_model.pt',
    'MQA': '/content/drive/MyDrive/GPTNeo_MQA_TinyStories/gptneo_mqa_tinystories/best_model.pt',
    'GQA-4': '/content/drive/MyDrive/GPTNeo_GQA4_TinyStories/gptneo_gqa4_tinystories/best_model.pt',
    'MLA': '/content/drive/MyDrive/GPTNeo_MLA_TinyStories/gptneo_mla_tinystories/best_model.pt',
}

# SimpleStories models
ss_paths = {
    'MHA': '/content/drive/MyDrive/GPTNeo_SimpleStories/best_model_mha_ss.pt',
    'MQA': '/content/drive/MyDrive/GPTNeo_SimpleStories/best_model_mqa_ss.pt',
    'GQA-4': '/content/drive/MyDrive/GPTNeo_SimpleStories/best_model_gqa_ss.pt',
    'MLA': '/content/drive/MyDrive/GPTNeo_SimpleStories/best_model_mla_ss.pt',
}

# Load all models
models = {'TinyStories': {}, 'SimpleStories': {}}
checkpoints = {'TinyStories': {}, 'SimpleStories': {}}

print("Loading TinyStories models...")
for name, path in ts_paths.items():
    try:
        models['TinyStories'][name], checkpoints['TinyStories'][name] = load_model(name, path)
        print(f"  {name}: loaded")
    except Exception as e:
        print(f"  {name}: FAILED - {e}")

print("\nLoading SimpleStories models...")
for name, path in ss_paths.items():
    try:
        models['SimpleStories'][name], checkpoints['SimpleStories'][name] = load_model(name, path)
        print(f"  {name}: loaded")
    except Exception as e:
        print(f"  {name}: FAILED - {e}")

print(f"\nLoaded {sum(len(v) for v in models.values())} / 8 models")

## 3. Parameter Count Comparison

In [ ]:
print("Parameter Count Comparison")
print("=" * 70)
print(f"{'Model':<10} {'Total Params':>15} {'Non-Embed':>15} {'Embed':>15}")
print("-" * 70)

for name in ['MHA', 'MQA', 'GQA-4', 'MLA']:
    for dataset in ['TinyStories', 'SimpleStories']:
        if name in models[dataset]:
            model = models[dataset][name]
            total = model.get_num_params()
            non_embed = model.get_num_params(non_embedding=True)
            embed = total - non_embed
            print(f"{name:<10} {total:>15,} {non_embed:>15,} {embed:>15,}")
            break  # Same architecture, only need one

print("=" * 70)

## 4. KV-Cache Analysis

In [ ]:
d_head = 256 // 8  # = 32
h = 8

kv_cache = {
    'MHA': 2 * h * d_head,        # 512 values per token per layer
    'MQA': 2 * 1 * d_head,        # 64 values
    'GQA-4': 2 * 4 * d_head,      # 256 values
    'MLA': 128 + 16,               # 144 values (d_c + d_rope)
}

print("KV-Cache Size per Token per Layer")
print("=" * 50)
for name, size in kv_cache.items():
    relative = size / kv_cache['MHA']
    print(f"  {name:<8}: {size:>4} values  ({relative:.3f}x of MHA)")

# Bar chart
fig, ax = plt.subplots(figsize=(8, 5))
names = list(kv_cache.keys())
sizes = list(kv_cache.values())
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']
bars = ax.bar(names, sizes, color=colors)
ax.set_ylabel('Values per Token per Layer')
ax.set_title('KV-Cache Size Comparison')
for bar, size in zip(bars, sizes):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5,
            str(size), ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig('/content/kv_cache_comparison.png', dpi=150)
plt.show()

## 5. Validation Perplexity

In [ ]:
from AttentionHeads.mha.data_loader import TinyStoriesDataModule

def evaluate_model(model, data_config, max_batches=100):
    """Evaluate model on validation set, return avg loss and perplexity"""
    data_cfg = {
        'dataset_name': data_config['dataset_name'],
        'tokenizer': 'gpt2',
        'train_samples': 1000,
        'val_samples': 5000,
        'batch_size': 32,
        'max_seq_length': 256,
        'num_workers': 2,
        'pin_memory': True,
        'text_column': data_config.get('text_column', 'text'),
        'val_split': data_config.get('val_split', 'validation'),
    }
    dm = TinyStoriesDataModule(data_cfg)
    dm.setup()
    val_loader = dm.val_dataloader()
    
    import torch.nn.functional as F
    total_loss = 0
    total_tokens = 0
    
    with torch.no_grad():
        for i, batch in enumerate(val_loader):
            if i >= max_batches:
                break
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            
            logits = model(input_ids)
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = input_ids[:, 1:].contiguous()
            shift_mask = attention_mask[:, 1:].contiguous()
            
            loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)),
                                   shift_labels.view(-1), reduction='none')
            loss = (loss * shift_mask.view(-1)).sum()
            total_loss += loss.item()
            total_tokens += shift_mask.sum().item()
    
    avg_loss = total_loss / total_tokens
    perplexity = np.exp(avg_loss)
    return avg_loss, perplexity

# Data configs
data_configs = {
    'TinyStories': {'dataset_name': 'roneneldan/TinyStories', 'text_column': 'text', 'val_split': 'validation'},
    'SimpleStories': {'dataset_name': 'SimpleStories/SimpleStories', 'text_column': 'story', 'val_split': 'test'},
}

# Evaluate all models
results = {}
for dataset in ['TinyStories', 'SimpleStories']:
    results[dataset] = {}
    print(f"\nEvaluating on {dataset}...")
    for name in ['MHA', 'MQA', 'GQA-4', 'MLA']:
        if name in models[dataset]:
            loss, ppl = evaluate_model(models[dataset][name], data_configs[dataset])
            results[dataset][name] = {'loss': loss, 'perplexity': ppl}
            print(f"  {name}: Loss={loss:.4f}, PPL={ppl:.2f}")

print("\nPerplexity Summary")
print("=" * 60)
print(f"{'Model':<10} {'TinyStories PPL':>18} {'SimpleStories PPL':>18}")
print("-" * 60)
for name in ['MHA', 'MQA', 'GQA-4', 'MLA']:
    ts_ppl = results.get('TinyStories', {}).get(name, {}).get('perplexity', 'N/A')
    ss_ppl = results.get('SimpleStories', {}).get(name, {}).get('perplexity', 'N/A')
    ts_str = f"{ts_ppl:.2f}" if isinstance(ts_ppl, float) else ts_ppl
    ss_str = f"{ss_ppl:.2f}" if isinstance(ss_ppl, float) else ss_ppl
    print(f"{name:<10} {ts_str:>18} {ss_str:>18}")

## 6. Top-1 and Top-5 Accuracy

In [ ]:
def compute_accuracy(model, data_config, max_batches=100):
    """Compute top-1 and top-5 next-token prediction accuracy"""
    data_cfg = {
        'dataset_name': data_config['dataset_name'],
        'tokenizer': 'gpt2',
        'train_samples': 1000,
        'val_samples': 5000,
        'batch_size': 32,
        'max_seq_length': 256,
        'num_workers': 2,
        'pin_memory': True,
        'text_column': data_config.get('text_column', 'text'),
        'val_split': data_config.get('val_split', 'validation'),
    }
    dm = TinyStoriesDataModule(data_cfg)
    dm.setup()
    val_loader = dm.val_dataloader()
    
    top1_correct = 0
    top5_correct = 0
    total = 0
    
    with torch.no_grad():
        for i, batch in enumerate(val_loader):
            if i >= max_batches:
                break
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            
            logits = model(input_ids)
            shift_logits = logits[:, :-1, :]
            shift_labels = input_ids[:, 1:]
            shift_mask = mask[:, 1:]
            
            # Top-1
            preds = shift_logits.argmax(dim=-1)
            top1_correct += ((preds == shift_labels) * shift_mask).sum().item()
            
            # Top-5
            _, top5 = shift_logits.topk(5, dim=-1)
            top5_match = (top5 == shift_labels.unsqueeze(-1)).any(dim=-1)
            top5_correct += (top5_match * shift_mask).sum().item()
            
            total += shift_mask.sum().item()
    
    return top1_correct / total * 100, top5_correct / total * 100

# Compute accuracies
acc_results = {}
for dataset in ['TinyStories', 'SimpleStories']:
    acc_results[dataset] = {}
    print(f"\nComputing accuracy on {dataset}...")
    for name in ['MHA', 'MQA', 'GQA-4', 'MLA']:
        if name in models[dataset]:
            top1, top5 = compute_accuracy(models[dataset][name], data_configs[dataset])
            acc_results[dataset][name] = {'top1': top1, 'top5': top5}
            print(f"  {name}: Top-1={top1:.2f}%, Top-5={top5:.2f}%")

print("\nAccuracy Summary")
print("=" * 80)
print(f"{'Model':<10} {'TS Top-1':>10} {'TS Top-5':>10} {'SS Top-1':>10} {'SS Top-5':>10}")
print("-" * 80)
for name in ['MHA', 'MQA', 'GQA-4', 'MLA']:
    ts = acc_results.get('TinyStories', {}).get(name, {})
    ss = acc_results.get('SimpleStories', {}).get(name, {})
    print(f"{name:<10} {ts.get('top1', 0):.2f}%{'':<4} {ts.get('top5', 0):.2f}%{'':<4} {ss.get('top1', 0):.2f}%{'':<4} {ss.get('top5', 0):.2f}%")

## 7. Generation Quality

In [ ]:
prompts = ["Once upon a time", "One day, a little girl", "In a big forest", "There was a happy dog"]

for dataset in ['TinyStories', 'SimpleStories']:
    print(f"\n{'='*80}")
    print(f"Generated Stories ({dataset})")
    print(f"{'='*80}")
    
    for prompt in prompts:
        print(f"\nPrompt: \"{prompt}\"")
        print("-" * 80)
        
        for name in ['MHA', 'MQA', 'GQA-4', 'MLA']:
            if name in models[dataset]:
                model = models[dataset][name]
                input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
                with torch.no_grad():
                    output = model.generate(input_ids, max_length=100, temperature=0.8, top_k=50, top_p=0.95)
                text = tokenizer.decode(output[0], skip_special_tokens=True)
                print(f"\n  [{name}]: {text[:200]}")
        print()

## 8. Inference Speed

In [ ]:
def benchmark_speed(model, num_runs=10, gen_length=100):
    """Benchmark generation speed in tokens/sec"""
    input_ids = tokenizer.encode("Once upon a time", return_tensors='pt').to(device)
    
    # Warmup
    with torch.no_grad():
        model.generate(input_ids, max_length=gen_length, temperature=0.8)
    
    torch.cuda.synchronize()
    times = []
    for _ in range(num_runs):
        torch.cuda.synchronize()
        start = time.time()
        with torch.no_grad():
            output = model.generate(input_ids, max_length=gen_length, temperature=0.8)
        torch.cuda.synchronize()
        elapsed = time.time() - start
        tokens_generated = output.shape[1] - input_ids.shape[1]
        times.append(tokens_generated / elapsed)
    
    return np.mean(times), np.std(times)

speed_results = {}
for name in ['MHA', 'MQA', 'GQA-4', 'MLA']:
    # Use TinyStories model (or SimpleStories if not available)
    for dataset in ['TinyStories', 'SimpleStories']:
        if name in models[dataset]:
            mean_speed, std_speed = benchmark_speed(models[dataset][name])
            speed_results[name] = {'mean': mean_speed, 'std': std_speed}
            print(f"{name}: {mean_speed:.1f} +/- {std_speed:.1f} tokens/sec")
            break

# Bar chart
fig, ax = plt.subplots(figsize=(8, 5))
names = list(speed_results.keys())
means = [speed_results[n]['mean'] for n in names]
stds = [speed_results[n]['std'] for n in names]
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']
bars = ax.bar(names, means, yerr=stds, color=colors, capsize=5)
ax.set_ylabel('Tokens / Second')
ax.set_title('Inference Speed Comparison')
plt.tight_layout()
plt.savefig('/content/inference_speed.png', dpi=150)
plt.show()

## 9. Peak GPU Memory

In [ ]:
memory_results = {}

for name in ['MHA', 'MQA', 'GQA-4', 'MLA']:
    for dataset in ['TinyStories', 'SimpleStories']:
        if name in models[dataset]:
            model = models[dataset][name]
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.empty_cache()
            
            input_ids = tokenizer.encode("Once upon a time", return_tensors='pt').to(device)
            with torch.no_grad():
                output = model.generate(input_ids, max_length=100, temperature=0.8)
            
            peak_mb = torch.cuda.max_memory_allocated() / (1024 * 1024)
            memory_results[name] = peak_mb
            print(f"{name}: {peak_mb:.1f} MB peak GPU memory")
            break

# Bar chart
fig, ax = plt.subplots(figsize=(8, 5))
names = list(memory_results.keys())
mems = [memory_results[n] for n in names]
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']
bars = ax.bar(names, mems, color=colors)
ax.set_ylabel('Peak GPU Memory (MB)')
ax.set_title('Peak GPU Memory During Generation')
for bar, mem in zip(bars, mems):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
            f'{mem:.0f}', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig('/content/peak_memory.png', dpi=150)
plt.show()

## 10. Training Curves

In [ ]:
from tensorboard.backend.event_processing import event_accumulator

# Log directories
log_dirs = {
    'TinyStories': {
        'MHA': '/content/logs/gptneo_mha_tinystories',
        'MQA': '/content/logs/gptneo_mqa_tinystories',
        'GQA-4': '/content/logs/gptneo_gqa4_tinystories',
        'MLA': '/content/logs/gptneo_mla_tinystories',
    },
    'SimpleStories': {
        'MHA': '/content/logs/gptneo_mha_simplestories',
        'MQA': '/content/logs/gptneo_mqa_simplestories',
        'GQA-4': '/content/logs/gptneo_gqa4_simplestories',
        'MLA': '/content/logs/gptneo_mla_simplestories',
    }
}

colors = {'MHA': '#2196F3', 'MQA': '#4CAF50', 'GQA-4': '#FF9800', 'MLA': '#E91E63'}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for row, dataset in enumerate(['TinyStories', 'SimpleStories']):
    for name, log_dir in log_dirs[dataset].items():
        try:
            ea = event_accumulator.EventAccumulator(log_dir)
            ea.Reload()
            
            # Training loss
            train_loss = ea.Scalars('train/loss')
            axes[row, 0].plot([x.step for x in train_loss], [x.value for x in train_loss],
                             label=name, color=colors[name], alpha=0.7)
            
            # Val loss
            val_loss = ea.Scalars('val/loss')
            axes[row, 1].plot([x.step for x in val_loss], [x.value for x in val_loss],
                             label=name, color=colors[name], marker='o', markersize=4)
        except Exception as e:
            print(f"Could not load {name} {dataset} logs: {e}")
    
    axes[row, 0].set_title(f'{dataset} - Training Loss')
    axes[row, 0].set_xlabel('Step')
    axes[row, 0].set_ylabel('Loss')
    axes[row, 0].legend()
    axes[row, 0].grid(True, alpha=0.3)
    
    axes[row, 1].set_title(f'{dataset} - Validation Loss')
    axes[row, 1].set_xlabel('Step')
    axes[row, 1].set_ylabel('Loss')
    axes[row, 1].legend()
    axes[row, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/training_curves_all.png', dpi=150)
plt.show()

## 11. Summary Tables

In [ ]:
print("COMPREHENSIVE COMPARISON: All 4 Attention Mechanisms")
print("=" * 100)

# Per-dataset tables
for dataset in ['TinyStories', 'SimpleStories']:
    print(f"\n{dataset}")
    print("-" * 100)
    print(f"{'Model':<10} {'Params':>10} {'PPL':>8} {'Top-1':>8} {'Top-5':>8} {'KV Cache':>10} {'Speed':>12} {'Memory':>10}")
    print("-" * 100)
    
    for name in ['MHA', 'MQA', 'GQA-4', 'MLA']:
        params = '-'
        ppl = '-'
        top1 = '-'
        top5 = '-'
        kv = str(kv_cache.get(name, '-'))
        speed = '-'
        mem = '-'
        
        if name in models.get(dataset, {}):
            params = f"{models[dataset][name].get_num_params():,}"
        if name in results.get(dataset, {}):
            ppl = f"{results[dataset][name]['perplexity']:.2f}"
        if name in acc_results.get(dataset, {}):
            top1 = f"{acc_results[dataset][name]['top1']:.1f}%"
            top5 = f"{acc_results[dataset][name]['top5']:.1f}%"
        if name in speed_results:
            speed = f"{speed_results[name]['mean']:.1f} tok/s"
        if name in memory_results:
            mem = f"{memory_results[name]:.0f} MB"
        
        print(f"{name:<10} {params:>10} {ppl:>8} {top1:>8} {top5:>8} {kv:>10} {speed:>12} {mem:>10}")

print("\n" + "=" * 100)
print("\nCross-Dataset Comparison: Do rankings change?")
print("-" * 60)
for name in ['MHA', 'MQA', 'GQA-4', 'MLA']:
    ts_ppl = results.get('TinyStories', {}).get(name, {}).get('perplexity', float('inf'))
    ss_ppl = results.get('SimpleStories', {}).get(name, {}).get('perplexity', float('inf'))
    if isinstance(ts_ppl, float) and isinstance(ss_ppl, float):
        print(f"  {name}: TS={ts_ppl:.2f}, SS={ss_ppl:.2f}")

## 12. Conclusion Charts

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
model_names = ['MHA', 'MQA', 'GQA-4', 'MLA']
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']

# 1. Perplexity comparison
ax = axes[0, 0]
for i, dataset in enumerate(['TinyStories', 'SimpleStories']):
    ppls = [results.get(dataset, {}).get(n, {}).get('perplexity', 0) for n in model_names]
    x = np.arange(len(model_names))
    width = 0.35
    ax.bar(x + i*width, ppls, width, label=dataset, alpha=0.8)
ax.set_ylabel('Perplexity')
ax.set_title('Validation Perplexity')
ax.set_xticks(x + width/2)
ax.set_xticklabels(model_names)
ax.legend()
ax.grid(True, alpha=0.3)

# 2. KV Cache
ax = axes[0, 1]
kv_sizes = [kv_cache[n] for n in model_names]
ax.bar(model_names, kv_sizes, color=colors)
ax.set_ylabel('Values per Token per Layer')
ax.set_title('KV-Cache Size')
ax.grid(True, alpha=0.3)

# 3. Speed
ax = axes[1, 0]
speeds = [speed_results.get(n, {}).get('mean', 0) for n in model_names]
ax.bar(model_names, speeds, color=colors)
ax.set_ylabel('Tokens / Second')
ax.set_title('Inference Speed')
ax.grid(True, alpha=0.3)

# 4. Memory
ax = axes[1, 1]
mems = [memory_results.get(n, 0) for n in model_names]
ax.bar(model_names, mems, color=colors)
ax.set_ylabel('Peak GPU Memory (MB)')
ax.set_title('Peak GPU Memory')
ax.grid(True, alpha=0.3)

plt.suptitle('Attention Mechanism Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/comparison_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print("Summary charts saved to /content/comparison_summary.png")

## Key Findings

### Attention Mechanisms Compared
| Feature | MHA | MQA | GQA-4 | MLA |
|---------|-----|-----|-------|-----|
| KV Cache | 512 (1.0x) | 64 (0.125x) | 256 (0.5x) | 144 (0.28x) |
| Quality | Baseline | Lower | Good | Close to MHA |
| Speed | Baseline | Fastest | Fast | Moderate |

### References
- MHA: Vaswani et al. (2017) - arXiv:1706.03762
- MQA: Shazeer (2019) - arXiv:1911.02150
- GQA: Ainslie et al. (2023) - arXiv:2305.13245
- MLA: DeepSeek-AI (2024) - arXiv:2405.04434
- RoPE: Su et al. (2021) - arXiv:2104.09864
- TinyStories: Eldan & Li (2023) - arXiv:2305.07759
- SimpleStories: Finke et al. (2025) - arXiv:2504.09184